# Review 1 Demo: Compiler-Inspired Relationship Extraction & Contextual Coherence
**Scope:** Review 1 (Modules 1, 2, and Basic 4)  
**Pipeline:** Batch execution without frontend; spaCy (en_core_web_md), NetworkX, Seaborn, pandas.


## 1. Module 1: Ingestion & Lexical Analysis (Token Table)

In [ ]:
import sys, os
import pandas as pd
import spacy

sys.path.insert(0, os.path.abspath(".."))
from engine.module1_lexical import lexical_analysis, clean_text, load_paragraphs

sentence = "Metformin reduces blood glucose in patients with type 2 diabetes."
doc, token_df = lexical_analysis(sentence)
display(token_df)


## 2. Module 2: Syntactic Dependency Tree & Active/Passive Voice Normalization

In [ ]:
from spacy import displacy
from engine.module2_triplets import extract_triplets

nlp = spacy.load("en_core_web_md")
active_sent = "Metformin reduces blood glucose."
passive_sent = "Blood glucose is reduced by metformin."

d_act = nlp(active_sent)
d_pass = nlp(passive_sent)

t_act = extract_triplets(d_act)
t_pass = extract_triplets(d_pass)

print("Active sentence:", active_sent)
print("  Extracted Triplet:", t_act)
print("Passive sentence:", passive_sent)
print("  Extracted Triplet:", t_pass)
print("Normalized Triplet Identity:", t_act == t_pass)

# Display dependency tree
displacy.render(d_act, style="dep", jupyter=True, options={"distance": 110})


## 3. Module 2: Negation-Aware Predicate Extraction

In [ ]:
neg_sentence = "Insulin does not cure diabetes."
d_neg = nlp(neg_sentence)
t_neg = extract_triplets(d_neg)

print("Sentence:", neg_sentence)
print("Extracted Negated Triplet:", t_neg)


## 4. Module 2: Unit Test Suite Summary

In [ ]:
from tests.test_module2 import run_and_report

test_results = run_and_report()
df_tests = pd.DataFrame(test_results)
display(df_tests[["ID", "Category", "Status", "Diagnosis"]])


## 5. Module 4: Jaccard vs Hybrid Score Comparison Table

In [ ]:
from engine.module4_scoring import compute_jaccard_similarity, compute_hybrid_score

pairs = [
    ("Active vs Passive", "Metformin reduces blood glucose.", "Blood glucose is reduced by metformin.", "Syntactic inversion with identical semantics"),
    ("Synonyms (AI vs ML)", "AI algorithms predict patient outcomes.", "Artificial intelligence models forecast clinical results.", "Different vocabulary with shared semantics"),
    ("Unrelated Pair", "Metformin reduces blood glucose.", "Solar panels generate green electricity.", "Zero thematic or structural overlap"),
    ("Negated Pair", "Insulin lowers blood glucose.", "Insulin does not lower blood glucose.", "Lexical similarity high, but polar negation present")
]

rows = []
for name, s1, s2, note in pairs:
    d1, d2 = nlp(s1), nlp(s2)
    jac = compute_jaccard_similarity(d1, d2)
    hyb, ts, vs = compute_hybrid_score(d1, d2, use_sbert=False)
    rows.append({
        "Comparison Pair": name,
        "Sentence 1": s1,
        "Sentence 2": s2,
        "Jaccard (Baseline)": round(jac, 3),
        "Proposed Hybrid": round(hyb, 3),
        "Triplet Sim": round(ts, 3),
        "Vector Sim": round(vs, 3),
        "Linguistic Note": note
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df[["Comparison Pair", "Jaccard (Baseline)", "Proposed Hybrid", "Triplet Sim", "Vector Sim", "Linguistic Note"]])


## 6. Module 4: Paragraph Coherence Heatmap & Semantic Drift Log

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from engine.module4_scoring import build_coherence_matrix, detect_semantic_drift, DRIFT_THRESHOLD

with open("../data/ibuprofen_drift.txt", "r", encoding="utf-8") as f:
    paras = [p.strip() for p in f.read().split("\n\n") if len(p.strip().split()) >= 5]

M, docs, trips = build_coherence_matrix(paras, w_triplet=0.4, w_vector=0.6, use_sbert=False)
drift_events = detect_semantic_drift(M, threshold=DRIFT_THRESHOLD)

# Heatmap
labels = [f"P{i+1}" for i in range(len(M))]
plt.figure(figsize=(6, 5))
sns.heatmap(M, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, xticklabels=labels, yticklabels=labels, cbar_kws={'label': 'Hybrid Coherence Score'})
plt.title("Coherence Heatmap (Ibuprofen with Injected Solar Drift)", fontweight="bold")
plt.show()

# Drift Log
print("--- Semantic Drift Detection Log ---")
for event in drift_events:
    print(event["log_message"])


## 7. Knowledge Graph Visualization (NetworkX)

In [ ]:
import networkx as nx

G = nx.DiGraph()
for plist in trips:
    for s, p, o in plist:
        G.add_edge(s, o, label=p)

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, k=0.9, seed=42)
nx.draw(G, pos, with_labels=True, node_color="#cfe3ff", edge_color="#6c757d", node_size=2400, font_size=8, font_weight="bold", arrows=True, arrowsize=15)
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
plt.title("Compiler-Derived Entity-Relationship Knowledge Graph", fontweight="bold")
plt.show()
